In [1]:
from pathlib import Path
from datetime import datetime
from typing import List
from types import SimpleNamespace
import pickle
import numpy as np
import torch

from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list

from nequip.radial_embedding import InitialEmbedding
from nequip.e3nn_nequip import NequIP

from torch_geometric.data import Data
from nequip.mlp import MLP_c, MLP_l, MLP_n
from torch import nn
import torch.nn.functional as F

def get_scaler(dataset, use_prop_scaler = False, 
               scaler_path = None):
    # Load once to compute property scaler
    if scaler_path is None:
        lattice_scaler = get_scaler_from_data_list(
            dataset.cached_data,
            key='scaled_lattice')
        if use_prop_scaler:
            NotImplementedError("Not implemented the multi prop scaler yet.")
    else:
        lattice_scaler = torch.load(
            Path(scaler_path) / 'lattice_scaler.pt')
    return lattice_scaler


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# setting
device = torch.device("cpu")

# dataset
dataset = CHGNetDataset(path= '/home/zhongpc/chggen/data/perov_5/test_zpc.csv',
                        name = 'zpc_test',
                        prop_list = ['heat_ref', 'heat_all'],
                        )
lattice_scaler = get_scaler(dataset= dataset)
item = dataset[0]
x_ = item['crys_graph']
y_ = item['properties']
x_ = x_.to(device = device)
y_ = y_.to(device = device)

# model
chggen = CHGGen(lattice_scaler= lattice_scaler) # good
chggen.to(device = device)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 297.90it/s]
/home/zhongpc/chggen/chggen/common/data_utils.py:647: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/zhongpc/chggen/chggen/common/data_utils.py:615: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


CHGGen(
  (encoder): CHGNet_encoder(
    (composition_model): AtomRef(
      (fc): Linear(in_features=94, out_features=1, bias=False)
    )
    (graph_converter): CrystalGraphConverter(algorithm='fast', atom_graph_cutoff=5, bond_graph_cutoff=3)
    (atom_embedding): AtomEmbedding(
      (embedding): Embedding(94, 64)
    )
    (bond_basis_expansion): BondEncoder(
      (rbf_expansion_ag): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
      (rbf_expansion_bg): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
    )
    (bond_embedding): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_ag): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_bg): Linear(in_features=9, out_features=64, bias=False)
    (angle_basis_expansion): AngleEncoder(
      (fourier_expansion): Fourier()
    )
    (angle_embedding): Linear(in_features=9, out_features=64, bias=False)
    (atom_conv_layers): ModuleList(
      (0-3): 4 x AtomConv(


In [3]:
from chggen.pl_modules.model import build_mlp 

MAX_ATOMIC_NUM = 100
max_atoms = 20
latent_dim = 64
hidden_dim = 128
fc_num_layers = 2

mlp_num_atoms = build_mlp(latent_dim, hidden_dim,
                      fc_num_layers, max_atoms+1)

mlp_lattice = build_mlp(latent_dim, hidden_dim,
                      fc_num_layers, 6)

mlp_composition = build_mlp(latent_dim, hidden_dim,
                            fc_num_layers, MAX_ATOMIC_NUM)



In [4]:

# encode
mu, log_var, z = chggen.encode([x_, x_])

# # predict.
# composition = mlp_composition(z).detach()          # 1D tensor
# pred_lengths_and_angles = mlp_lattice(z).detach()              # 3D tensor
# num_atoms = mlp_num_atoms(z).detach()       # 1D tensor with max_atom + 1


# predict.
num_atoms_prob = chggen.predict_num_atoms(z).detach() 
num_atoms = chggen.predict_num_atoms(z).argmax(dim=-1) # 1D tensor with max_atom + 1
composition_per_atom = chggen.predict_composition(z, num_atoms).detach()          # 2D tensor [atom_batch, max_Z]
lengths_and_angles, lengths, angles = chggen.predict_lattice(z, num_atoms)              # 3D tensor

# pred_composition = chggen.sample_composition(composition_prob= composition_prob_per_atom, num_atoms= num_atoms)

/home/zhongpc/chggen/chggen/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [5]:
cur_atom_types = chggen.sample_composition(composition_prob= F.softmax(composition_per_atom, dim=-1), num_atoms= num_atoms)

In [6]:
cur_frac_coords = torch.rand((num_atoms.sum(), 3), device=z.device)

In [7]:
# cur_frac_coords

In [8]:
lengths # [N_crystal, 3]

tensor([[5.1098, 5.4244, 4.9762],
        [3.0125, 3.1379, 3.0617]])

In [9]:
angles # [N_crystal, 3]

tensor([[90.0000, 90.0000, 90.0000],
        [90.0000, 90.0000, 90.0000]])

In [11]:
num_atoms

tensor([10,  2])

In [12]:
num_atoms

tensor([10,  2])

In [13]:
batch = torch.arange(len(num_atoms))
batch = batch.repeat_interleave(num_atoms)

In [14]:
angles_ = angles.repeat_interleave(num_atoms, dim = 0)
lengths_ = lengths.repeat_interleave(num_atoms, dim = 0)

In [27]:
lengths_

tensor([[5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [5.1098, 5.4244, 4.9762],
        [3.0125, 3.1379, 3.0617],
        [3.0125, 3.1379, 3.0617]])

In [16]:
from torch_scatter import scatter

# frac_scatter = scatter(
#             cur_frac_coords, index= batch, dim=0,)


# for frac_coor, atom_type, length, angle in zip(cur_frac_coords, cur_atom_types, lengths_, angles_):
#     pass
    

In [31]:
from torch_geometric.data import Data
from chggen.common.data_utils import get_pbc_distances,radius_graph_pbc


cutoff = 6.0
max_neighbors = 12
edge_index, to_jimages, num_bonds = radius_graph_pbc(
                cur_frac_coords, lengths, angles, num_atoms, cutoff, max_neighbors,
                device=num_atoms.device)

In [35]:
out = get_pbc_distances(
            cur_frac_coords,
            edge_index,
            lengths,
            angles,
            to_jimages,
            num_atoms,
            num_bonds,
            coord_is_cart=True,
            return_offsets=True,
            return_distance_vec=True,
        )

In [38]:
out.keys()

dict_keys(['edge_index', 'distances', 'distance_vec', 'offsets'])

In [87]:
from ase import Atoms

all_cells = []
for length, angle in zip(lengths, angles):
    params = torch.cat((length, angle), dim = 0)
    atoms = Atoms(cell=params, pbc=True)
    cell = atoms.get_cell() 
    all_cells.append(cell)

all_cells = torch.tensor(all_cells)
all_cells = all_cells.repeat_interleave(num_atoms, dim = 0)



In [86]:
edge_index.shape

torch.Size([2, 144])

In [95]:
data = Data(
            x       = cur_atom_types,
            pos     = cur_frac_coords,
            cell    = all_cells,
            pbc     = True,
            edge_index = edge_index,
            edge_attr = out['distance_vec']
        )

In [96]:
max_Z = 100
cutoff = 6.0
nequip = NequIP(init_embed     = InitialEmbedding(num_species= max_Z, cutoff=cutoff),
                        irreps_node_x  = '8x0e',
                        irreps_node_z  = '8x0e',
                        irreps_hidden  = '8x0e + 8x1e + 4x2e',
                        irreps_edge    = '1x0e + 1x1e + 1x2e',
                        irreps_out     = '1x1e',
                        num_convs      = 3,
                        radial_neurons = [16, 64],
                        num_neighbors  = 12,
                    )

/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


In [98]:
pred_displace_nequip = nequip(data)

In [101]:
pred_displace_nequip

tensor([[-3.5847e-01,  9.3114e-01,  9.8191e-01],
        [-2.8726e-02,  7.4296e-02, -1.5332e-02],
        [-9.0865e-01,  2.6169e-01,  2.5994e-01],
        [-9.3830e-02,  3.5732e-01,  1.6392e-01],
        [ 4.8126e-02,  6.6448e-02, -1.4198e-01],
        [-5.8239e-04, -4.5627e-04,  1.4073e-03],
        [-2.6284e-02, -2.0598e-03, -1.5603e-02],
        [ 1.2654e+00,  5.4716e+00, -1.5616e+00],
        [-1.8769e-01,  1.2524e-01, -1.6482e-01],
        [ 3.3146e-02,  2.1538e-02,  1.1239e-01],
        [ 7.0608e-02, -2.2472e-02,  2.0921e-01],
        [-5.3899e-02, -1.8695e-02, -1.4001e-01]],
       grad_fn=<ReshapeAliasBackward0>)

In [112]:
from chggen.common.data_utils import frac_to_cart_coords, cart_to_frac_coords

In [107]:
cur_pos_cart = frac_to_cart_coords(cur_frac_coords, lengths, angles, num_atoms)

In [114]:
pred_pos_cart = cur_pos_cart + pred_displace_nequip
pred_frac_coords = cart_to_frac_coords(pred_pos_cart, lengths, angles, num_atoms)

In [115]:
pred_frac_coords

tensor([[0.4389, 0.9779, 0.3391],
        [0.2729, 0.2056, 0.3292],
        [0.8701, 0.2270, 0.7060],
        [0.5997, 0.5437, 0.0512],
        [0.8488, 0.9562, 0.4470],
        [0.5685, 0.3700, 0.1928],
        [0.1632, 0.6394, 0.1289],
        [0.0088, 0.5234, 0.5153],
        [0.6209, 0.8064, 0.6099],
        [0.2497, 0.0391, 0.6537],
        [0.4902, 0.5163, 0.5435],
        [0.6947, 0.4181, 0.8991]], grad_fn=<RemainderBackward0>)

In [45]:

# random initialize.
nums_atoms =  torch.round(composition*num_atoms)
atom_type = 0
x = []
for num in nums_atoms:
    x_temp = [atom_type] * num 
x += x_temp                             # node feature, atomic type here
pos = torch.rand(size=(round(num_atoms), 3))   # atoms position
cell = lattice                          # cell size 
pbc = True                              # periodic boundary condition


In [14]:
mlp_num_atoms(z).detach().shape

torch.Size([2, 21])

In [13]:
mlp_lattice(z).detach()  

tensor([[-0.0673, -0.1879,  0.0574,  0.1654, -0.0310,  0.1439],
        [-0.1578, -0.1872,  0.2311, -0.1533,  0.1420, -0.0115]])

In [ ]:

data = Data(
    x = torch.tensor(x).long(),
    pos = torch.tensor(pos).float(),
    cell = torch.tensor(cell).float(),
    pbc = torch.tensor(pbc).bool(),
)

# denoise
nequip(data)

ld_kwargs = SimpleNamespace(n_step_each = 100,
                                step_lr = 1e-4,
                                min_sigma = 0,
                                save_traj = False,
                                disable_bar = False)

output = chggen.langevin_dynamics(z= z,
                                  ld_kwargs= ld_kwargs)

with open('./output_from_ld', 'wb') as fp:
    pickle.dump(output, fp)

print("Done")



## 
# data_params = {
#     "batch_size": 16,
#     "pin_memory": True,
#     "shuffle": True,
#     "collate_fn": collate_batch_v1,
# }

# data_loader = DataLoader(dataset, **data_params